In [ ]:
import os
os.environ['PYTHONSAFEPATH'] = '1'
from sklearn.datasets import fetch_20newsgroups
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))


newsgroup = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
documents = newsgroup.data

print(f"Number of documents = {len(documents)}")
print(documents[0][:500])

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) >= 3]
    return tokens

p_docs = [preprocess(doc) for doc in documents]  # p_docs = processed docs
print(p_docs[0][:20])

avg_length = sum(len(doc) for doc in p_docs)/len(p_docs)
print("Average Document Length: ", avg_length)

docs = [doc for doc in p_docs if len(doc)>0]
print(f"Number of documents after cleaning: {len(docs)}")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/requested/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Number of documents = 18846


I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a
['sure', 'bashers', 'pens', 'fans', 'pretty', 'confused', 'lack', 'kind', 'posts', 'recent', 'pens', 'massacre', 'devils', 'actually', 'bit', 'puzzled', 'bit', 'relieved', 'however', 'going']
Average Document Length:  109.21171601400827
Number of documents after cleaning: 18302


Buliding Vocabulary:

In [2]:
from collections import Counter

word_counts = Counter(word for doc in docs for word in doc)

print(f"Most common words are: {word_counts.most_common(10)}")
print(f"Least common words are: {word_counts.most_common()[-10:]}")
print(f"Total unique words are: {len(word_counts)}")

Most common words are: [('ax', 62507), ('x', 14196), ('one', 10409), ('would', 10191), ('q', 9392), ('w', 9068), ('f', 8189), ('b', 8168), ('g', 8113), ('p', 7988)]
Least common words are: [('sured', 1), ('intracranial', 1), ('paroxysmal', 1), ('hemicrania', 1), ('aneurysm', 1), ('recepticles', 1), ('equi', 1), ('fcom', 1), ('pontaic', 1), ('definate', 1)]
Total unique words are: 94135


In [3]:
min_frequency = 4
vocabulary = {word for word, count in word_counts.items() if count >= min_frequency}
print(f"Vocabulary size after filtering: {len(vocabulary)}")
print(vocabulary)

Vocabulary size after filtering: 30440
{'advancement', 'destrade', 'reboost', 'bates', 'wore', 'defiled', 'talent', 'abolition', 'terrain', 'mn', 'unixware', 'date', 'stroke', 'sync', 'alloy', 'accuse', 'requirements', 'weird', 'inert', 'receive', 'conventions', 'offensive', 'insulted', 'allez', 'wfw', 'lockup', 'xman', 'reminded', 'stipulate', 'newsletter', 'montgomery', 'astronomer', 'aligning', 'icj', 'bat', 'spawned', 'attacking', 'favourite', 'noyes', 'deluded', 'newsline', 'ih', 'injustices', 'gutter', 'booked', 'bbbbb', 'waikato', 'toxic', 'noone', 'demonstrators', 'hangar', 'gut', 'andrews', 'reeds', 'setjmp', 'anania', 'rdf', 'eij', 'bosnian', 'golly', 'falloon', 'policies', 'landmark', 'scandinavian', 'scramble', 'arcs', 'ment', 'oasis', 'curses', 'cobralinks', 'ursa', 'rafik', 'vat', 'macdonald', 'lopez', 'conversation', 'escalating', 'rendezvous', 'insanity', 'shadowing', 'defensively', 'reduces', 'contextual', 'candidiasis', 'rlogin', 'blvd', 'carpenter', 'spi', 'constrain

word2idx and idx2word mappings:

In [4]:
word2idx = {word: idx for idx, word in enumerate(sorted(vocabulary))}
idx2word = {idx: word for word, idx in word2idx.items()}

In [5]:
# Generating pairs for training for word2vec
def generate_training_pairs(docs, word2idx, window_size=2):
    pairs = []
    for doc in docs:
        doc = [word for word in doc if word in word2idx]

        for i, target_word in enumerate(doc):
            target_idx = word2idx[target_word]

            left = max(0, i-window_size)
            right = min(len(doc), i+window_size+1)

            for j in range(left, right):
                if i == j:
                    continue
                context_word = doc[j]
                context_idx = word2idx[context_word]
                pairs.append((target_idx, context_idx))

    return pairs

In [6]:
pairs = generate_training_pairs(docs, word2idx, window_size=2)

print(f"Total training pairs = {len(pairs)}")
print(f"First 5 pairs: {pairs[:5]}")

for target_idx, context_idx in pairs[:5]:
    print(f"{idx2word[target_idx]} -> {idx2word[context_idx]}")

Total training pairs = 7764228
First 5 pairs: [(26018, 2273), (26018, 19626), (2273, 26018), (2273, 19626), (2273, 9497)]
sure -> bashers
sure -> pens
bashers -> sure
bashers -> pens
bashers -> fans


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "mps" if torch.backends.mps.is_available() else "cpu"
vocab_size, embed_dim, num_neg = len(word2idx), 100, 5

In [ ]:


counts = torch.tensor([word_counts[idx2word[i]] for i in range(vocab_size)], dtype=torch.float)
noise_dist = (counts ** 0.75 / (counts ** 0.75).sum()).to(device)

class SkipGramNS(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, dim)  # target vectors
        self.out_embed = nn.Embedding(vocab_size, dim) # context vectors
        nn.init.uniform_(self.in_embed.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.out_embed.weight)
    
    def forward(self, target, context, negatives):
        v = self.in_embed(target)
        pos = (v * self.out_embed(context)).sum(1)
        neg = torch.bmm(self.out_embed(negatives), v.unsqueeze(2)).squeeze(2)
        return -(F.logsigmoid(pos) + F.logsigmoid(-neg).sum(1)).mean()

model = SkipGramNS(vocab_size, embed_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
pairs_t = torch.tensor(pairs, dtype=torch.long)
batch_size = 1024

for epoch in range(3):
    perm = torch.randperm(len(pairs_t))
    total = 0.0
    for i in range(0, len(perm), batch_size):
        batch = pairs_t[perm[i:i+batch_size]].to(device)
        target, context = batch[:, 0], batch[:, 1]
        negatives = torch.multinomial(noise_dist, len(batch) * num_neg, replacement=True).view(len(batch), num_neg)
        loss = model(target, context, negatives)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f" epoch {epoch}: loss = {total/(len(perm) // batch_size):.4f}")

emb = F.normalize(model.in_embed.weight.detach(), dim=1)
def most_similar(word, k=10):
    sims = emb @ emb[word2idx[word]]
    return [(idx2word[i], round(sims[i].item(), 3)) for i in sims.topk(k+1).indices.tolist()[1:]]



 epoch 0: loss = 2.3474
 epoch 1: loss = 2.0493
 epoch 2: loss = 1.9452
god [('righteousness', 0.825), ('deity', 0.795), ('omnipotent', 0.786), ('spirit', 0.782), ('omniscience', 0.78), ('creator', 0.778), ('obedience', 0.776), ('spirits', 0.773), ('whosoever', 0.77), ('supernatural', 0.766)]
car [('cars', 0.806), ('tire', 0.798), ('sporty', 0.773), ('servotronic', 0.771), ('dealership', 0.771), ('convertible', 0.76), ('towed', 0.759), ('truck', 0.757), ('trailer', 0.752), ('toyota', 0.752)]
space [('shuttle', 0.686), ('shuttles', 0.662), ('inflatable', 0.639), ('activist', 0.616), ('hubble', 0.613), ('ksc', 0.611), ('billboard', 0.61), ('payloads', 0.607), ('mir', 0.606), ('portuguese', 0.6)]
windows [('workgroups', 0.826), ('apps', 0.816), ('antivirus', 0.81), ('microsoft', 0.798), ('excel', 0.798), ('desktop', 0.796), ('pktmux', 0.789), ('telix', 0.784), ('excursion', 0.784), ('procomm', 0.78)]


In [ ]:
for w in ["god", "car", "space", "windows"]:
    print(w, most_similar(w))